In [0]:

from pyspark.sql import functions as F

CATALOG = "finance_management"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")


def publish_gold(df, table_name):
    target = f"{GOLD}.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )

    print(f"Published {target}: {df.count()} rows")


# --------------------------------------------------
# 1. Dimension tables
# --------------------------------------------------

customers = spark.table(f"{SILVER}.customers")

customer_dim = customers.select(
    "customer_id",
    "customer_name",
    "email",
    "country",
    "customer_type",
    "status",
    "created_date"
)

publish_gold(customer_dim, "dim_customer")


vendors = spark.table(f"{SILVER}.vendors")

vendor_dim = vendors.select(
    "vendor_id",
    "vendor_name",
    "email",
    "country",
    "vendor_category",
    "status",
    "created_date"
)

publish_gold(vendor_dim, "dim_vendor")


accounts = spark.table(f"{SILVER}.chart_of_accounts")

account_dim = accounts.select(
    "account_id",
    "account_name",
    "account_type",
    "account_category",
    "is_active"
)

publish_gold(account_dim, "dim_account")


# --------------------------------------------------
# 2. Fact tables
# --------------------------------------------------

revenue = (
    spark.table(f"{SILVER}.revenue_transactions")
    .withColumn("transaction_year", F.year("transaction_date"))
    .withColumn(
        "transaction_month",
        F.date_format("transaction_date", "yyyy-MM")
    )
)

publish_gold(revenue, "fact_revenue")


expenses = (
    spark.table(f"{SILVER}.expense_transactions")
    .withColumn("transaction_year", F.year("transaction_date"))
    .withColumn(
        "transaction_month",
        F.date_format("transaction_date", "yyyy-MM")
    )
)

publish_gold(expenses, "fact_expense")


invoices = spark.table(f"{SILVER}.invoices")

publish_gold(invoices, "fact_invoices")


budgets = spark.table(f"{SILVER}.budgets")

publish_gold(budgets, "fact_budgets")


journals = spark.table(f"{SILVER}.journal_entries")

publish_gold(journals, "fact_journal_entries")


# --------------------------------------------------
# 3. Monthly revenue summary
# --------------------------------------------------

monthly_revenue = (
    revenue
    .groupBy("transaction_month", "currency")
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.countDistinct("transaction_id").alias(
            "revenue_transaction_count"
        )
    )
)


# --------------------------------------------------
# 4. Monthly expense summary
# --------------------------------------------------

monthly_expense = (
    expenses
    .groupBy("transaction_month", "currency")
    .agg(
        F.sum("amount").alias("total_expenses"),
        F.countDistinct("transaction_id").alias(
            "expense_transaction_count"
        )
    )
)


# --------------------------------------------------
# 5. Profit and Loss summary
# --------------------------------------------------

# Revenue aggregation by year, month and currency
revenue_summary = (
    spark.table("finance_management.gold.fact_revenue")
    .withColumn(
        "transaction_year",
        F.date_format(F.to_date("transaction_date"), "yyyy")
    )
    .withColumn(
        "transaction_month",
        F.date_format(F.to_date("transaction_date"), "MM")
    )
    .groupBy(
        "transaction_year",
        "transaction_month",
        "currency"
    )
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.count("*").alias("revenue_transaction_count")
    )
)

# Expense aggregation by year, month and currency
expense_summary = (
    spark.table("finance_management.gold.fact_expense")
    .withColumn(
        "transaction_year",
        F.date_format(F.to_date("transaction_date"), "yyyy")
    )
    .withColumn(
        "transaction_month",
        F.date_format(F.to_date("transaction_date"), "MM")
    )
    .groupBy(
        "transaction_year",
        "transaction_month",
        "currency"
    )
    .agg(
        F.sum("amount").alias("total_expenses"),
        F.count("*").alias("expense_transaction_count")
    )
)

# Combine revenue and expenses
profit_loss_summary = (
    revenue_summary.alias("r")
    .join(
        expense_summary.alias("e"),
        on=[
            "transaction_year",
            "transaction_month",
            "currency"
        ],
        how="full_outer"
    )
    .select(
        "transaction_year",
        "transaction_month",
        "currency",
        F.coalesce(
            F.col("r.total_revenue"), F.lit(0)
        ).alias("total_revenue"),
        F.coalesce(
            F.col("r.revenue_transaction_count"), F.lit(0)
        ).alias("revenue_transaction_count"),
        F.coalesce(
            F.col("e.total_expenses"), F.lit(0)
        ).alias("total_expenses"),
        F.coalesce(
            F.col("e.expense_transaction_count"), F.lit(0)
        ).alias("expense_transaction_count")
    )
    .withColumn(
        "gross_profit",
        F.col("total_revenue") - F.col("total_expenses")
    )
    .withColumn(
        "profit_margin_percentage",
        F.when(
            F.col("total_revenue") != 0,
            (F.col("gross_profit") / F.col("total_revenue")) * 100
        ).otherwise(F.lit(0))
    )
    .orderBy(
        "transaction_year",
        "transaction_month",
        "currency"
    )
)

publish_gold(profit_loss_summary, "profit_loss_summary")


# --------------------------------------------------
# 6. Finance KPI summary
# --------------------------------------------------

finance_kpis = (
    profit_loss_summary
    .select(
        "transaction_month",
        "currency",
        "total_revenue",
        "total_expenses",
        "gross_profit",
        "profit_margin_percentage",
        "revenue_transaction_count",
        "expense_transaction_count"
    )
)

publish_gold(finance_kpis, "finance_kpi_summary")


# --------------------------------------------------
# 7. Accounts Receivable summary
# --------------------------------------------------

ar_summary = (
    invoices
    .groupBy("currency", "status")
    .agg(
        F.countDistinct("invoice_id").alias("invoice_count"),
        F.sum("invoice_amount").alias("invoice_amount"),
        F.sum("paid_amount").alias("paid_amount"),
        F.sum("outstanding_amount").alias(
            "outstanding_amount"
        )
    )
)

publish_gold(ar_summary, "accounts_receivable_summary")


# --------------------------------------------------
# 8. Accounts Payable summary
# --------------------------------------------------

ap_summary = (
    expenses
    .groupBy("currency")
    .agg(
        F.sum("amount").alias("total_expense_amount"),
        F.countDistinct("transaction_id").alias(
            "expense_count"
        )
    )
)

publish_gold(ap_summary, "accounts_payable_summary")


print("\nGold transformations completed successfully.")


In [0]:
%sql
SELECT *
FROM finance_management.gold.profit_loss_summary
ORDER BY transaction_year, transaction_month, currency;